# Credit Card Fraud Detection MLOps Pipeline using TFX
# Proyek 1: Membangun Pipeline Machine Learning, Credit card fraud detection
- **Nama:** Muhammad Fathurrohman
- 
- **ID Dicoding:** M_Fathurrohman

### Pipeline Steps:
1. **Ingest Data** (`CsvExampleGen`)
2. **Compute Data Statistics** (`StatisticsGen`)
3. **Infer Schema** (`SchemaGen`)
4. **Validate Data** (`ExampleValidator`)
5. **Preprocess Data** (`Transform`)
6. **Tune Hyperparameters** (`Tuner`)
7. **Train Model** (`Trainer`)
8. **Evaluate Model** (`Resolver` & `Evaluator`)
9. **Deploy Model** (`Pusher`)

## 1. Import Library and Dependencies

In [3]:
import os
import pandas as pd
import tensorflow as tf
import tensorflow_model_analysis as tfma
from keras_tuner import RandomSearch

# Importing TFX components
from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator
from tfx.components import Transform, Trainer, Tuner, Evaluator, Pusher
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2

# TFX Interactive Context
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

# Resolver for resolving inputs in pipelines
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy

# TFX standard artifact types
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

import urllib.request

## 2. Set Variables

In [4]:
PIPELINE_NAME = "cc-fraud-pipeline"
MODEL_NAME = "cc-fraud-model"

# Directory for storing generated artifacts
PIPELINE_ROOT = os.path.join('cc_fraud_pipeline_artifacts', PIPELINE_NAME)

# Path to SQLite DB file for MLMD storage
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')

# Output directory for exporting trained models
SERVING_MODEL_DIR = os.path.join('serving_model_dir', MODEL_NAME)

# Pipeline inputs
DATA_ROOT = "cc_data"
TRANSFORM_MODULE_FILE = "modules/cc_fraud_transform.py"
TRAINER_MODULE_FILE = "modules/cc_fraud_trainer.py"
TUNER_MODULE_FILE = "modules/cc_fraud_tuner.py"

# Create directories
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs("modules", exist_ok=True)

print("Pipeline Root:", PIPELINE_ROOT)
print("Serving Model Directory:", SERVING_MODEL_DIR)

Pipeline Root: cc_fraud_pipeline_artifacts\cc-fraud-pipeline
Serving Model Directory: serving_model_dir\cc-fraud-model


## 3. Interactive Context Initialization


In [5]:
interactive_context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

## 4. Download and Preprocess Dataset


In [6]:
import os
import requests
import pandas as pd

dataset_url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
csv_path = os.path.join(DATA_ROOT, "creditcard.csv")

if not os.path.exists(csv_path):
    print("Downloading Credit Card Fraud dataset...")
    response = requests.get(dataset_url, stream=True)
    response.raise_for_status()
    with open(csv_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    print("Download completed!")
else:
    print("Dataset already exists.")

df = pd.read_csv(csv_path)
print(f"Initial shape: {df.shape}")
print("Class distribution:\n", df["Class"].value_counts())
df.head()


Dataset already exists.
Initial shape: (284807, 31)
Class distribution:
 0    284315
1       492
Name: Class, dtype: int64


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 5. CsvExampleGen: Read and Split Dataset


In [7]:
output = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name='train', hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name='eval', hash_buckets=2)
    ])
)

example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output)
interactive_context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 10
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

## 6. StatisticsGen: Generate Data Statistics


In [8]:
statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
interactive_context.run(statistics_gen)
interactive_context.show(statistics_gen)

StatisticsGen(spec: <tfx.types.standard_component_specs.StatisticsGenSpec object at 0x000002CB9687FD90>, executor_spec: <tfx.dsl.components.base.executor_spec.BeamExecutorSpec object at 0x000002CB9687FA00>, driver_class: <class 'tfx.dsl.components.base.base_driver.BaseDriver'>, component_id: StatisticsGen, inputs: {'examples': OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={})}, outputs: {'statistics': OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={})})

## 7. SchemaGen: Generate Data Schema


In [9]:
schema_gen = SchemaGen(statistics=statistics_gen.outputs['statistics'], infer_feature_shape=True)
interactive_context.run(schema_gen)
interactive_context.show(schema_gen)

SchemaGen(spec: <tfx.types.standard_component_specs.SchemaGenSpec object at 0x000002CB9687F820>, executor_spec: <tfx.dsl.components.base.executor_spec.ExecutorClassSpec object at 0x000002CB9687F850>, driver_class: <class 'tfx.dsl.components.base.base_driver.BaseDriver'>, component_id: SchemaGen, inputs: {'statistics': OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={})}, outputs: {'schema': OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={})})

## 8. ExampleValidator: Validate Data Integrity


In [10]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
interactive_context.run(example_validator)
interactive_context.show(example_validator)

ExampleValidator(spec: <tfx.types.standard_component_specs.ExampleValidatorSpec object at 0x000002CBD7918100>, executor_spec: <tfx.dsl.components.base.executor_spec.ExecutorClassSpec object at 0x000002CBD7918820>, driver_class: <class 'tfx.dsl.components.base.base_driver.BaseDriver'>, component_id: ExampleValidator, inputs: {'statistics': OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}), 'schema': OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={})}, outputs: {'anomalies': OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={})})

## 9. Transform: Feature Engineering


In [11]:
%%writefile {TRANSFORM_MODULE_FILE}
import tensorflow as tf
import tensorflow_transform as tft

# List of numerical features to scale
NUMERICAL_FEATURES = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]
LABEL_KEY = "Class"

def transformed_name(key):
    """Renaming transformed features by appending _xf"""
    return key + "_xf"

def preprocessing_fn(inputs):
    """tf.transform's callback function for preprocessing inputs.
    Args:
        inputs: map from feature keys to RawTensors.
    Returns:
        outputs: map from feature keys to TransformedTensors.
    """
    outputs = {}
    
    # Standardize numerical features using Z-score scaling
    for key in NUMERICAL_FEATURES:
        outputs[transformed_name(key)] = tft.scale_to_z_score(inputs[key])
        
    # Pass through target label, casting to float32 for model compat
    outputs[transformed_name(LABEL_KEY)] = tf.cast(inputs[LABEL_KEY], tf.float32)
    
    return outputs

Overwriting modules/cc_fraud_transform.py


In [12]:
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE)
)
interactive_context.run(transform)

Instructions for updating:
Use ref() instead.


Instructions for updating:
Use ref() instead.


INFO:tensorflow:Assets written to: cc_fraud_pipeline_artifacts\cc-fraud-pipeline\Transform\transform_graph\14\.temp_path\tftransform_tmp\20b6e15c5d434f559ab36b4cb5eabc63\assets


INFO:tensorflow:Assets written to: cc_fraud_pipeline_artifacts\cc-fraud-pipeline\Transform\transform_graph\14\.temp_path\tftransform_tmp\20b6e15c5d434f559ab36b4cb5eabc63\assets


INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 14
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={})
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={})
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={})
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={})
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={})
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}))

## 10. Tuner: Hyperparameter Tuning


In [13]:
%%writefile {TUNER_MODULE_FILE}
import os
import tensorflow as tf
import tensorflow_transform as tft
from tensorflow.keras import layers
from keras_tuner.engine import base_tuner
from keras_tuner import RandomSearch
import keras_tuner as kt
from tfx.components.trainer.fn_args_utils import FnArgs
from typing import Any, Dict, NamedTuple, Text

NUMERICAL_FEATURES = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]
LABEL_KEY = "Class"

def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"

def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs=None,
             batch_size=128) -> tf.data.Dataset:
    """Get post-transform features & create batches of data"""
    
    # Get post-transform feature spec
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    # create batches of data
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    
    return dataset

# Model builder for hyperparameter tuning
def model_builder(hp):
    """Build machine learning model"""
    # Hyperparameters to tune
    num_layers = hp.Int('num_layers', min_value=1, max_value=3, step=1)
    dense_units = hp.Int('dense_units', min_value=32, max_value=128, step=32)
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5, step=0.1)
    learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4])

    # Inputs for all transformed numerical features
    inputs = {}
    for key in NUMERICAL_FEATURES:
        inputs[transformed_name(key)] = tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.float32)

    # Concatenate features
    x = tf.keras.layers.concatenate(list(inputs.values()))

    # Build dense layers
    for _ in range(num_layers):
        x = layers.Dense(dense_units, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout_rate)(x)

    # Output layer (sigmoid activation for binary classification)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    
    # Compile with AUC, Precision, Recall, and Accuracy
    model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model

TunerFnResult = NamedTuple('TunerFnResult', [
    ('tuner', base_tuner.BaseTuner),
    ('fit_kwargs', Dict[Text, Any]),
])

# Tuner function
def tuner_fn(fn_args: FnArgs):
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    
    # Determine steps
    train_steps = fn_args.train_steps if fn_args.train_steps and fn_args.train_steps > 0 else None
    eval_steps = fn_args.eval_steps if fn_args.eval_steps and fn_args.eval_steps > 0 else None

    # Load datasets (repeat if steps are specified, otherwise load once per epoch)
    train_set = input_fn(
        fn_args.train_files[0],
        tf_transform_output,
        num_epochs=None if train_steps else 1,
        batch_size=128
    )
    val_set = input_fn(
        fn_args.eval_files[0],
        tf_transform_output,
        num_epochs=None if eval_steps else 1,
        batch_size=128
    )

    model_tuner = RandomSearch(
        hypermodel=model_builder,
        objective=kt.Objective('val_auc', direction='max'),
        max_trials=3,
        executions_per_trial=1,
        directory=fn_args.working_dir,
        project_name='cc_fraud_tuner',
    )

    return TunerFnResult(
        tuner=model_tuner,
        fit_kwargs={
            'x': train_set,
            'validation_data': val_set,
            'steps_per_epoch': train_steps,
            'validation_steps': eval_steps,
            'callbacks': [
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_auc',
                    mode='max',
                    patience=2,
                    verbose=1
                )
            ]
        }
    )

Overwriting modules/cc_fraud_tuner.py


In [14]:
tuner = Tuner(
    module_file=os.path.abspath(TUNER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=100),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=50)
)
interactive_context.run(tuner)

Trial 3 Complete [00h 00m 06s]
val_auc: 0.996411919593811

Best val_auc So Far: 0.996411919593811
Total elapsed time: 00h 00m 22s
Results summary
Results in cc_fraud_pipeline_artifacts\cc-fraud-pipeline\.temp\15\cc_fraud_tuner
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 2 summary
Hyperparameters:
num_layers: 3
dense_units: 32
dropout_rate: 0.1
learning_rate: 0.001
Score: 0.996411919593811

Trial 1 summary
Hyperparameters:
num_layers: 2
dense_units: 96
dropout_rate: 0.0
learning_rate: 0.0001
Score: 0.8591673374176025

Trial 0 summary
Hyperparameters:
num_layers: 3
dense_units: 32
dropout_rate: 0.1
learning_rate: 0.0001
Score: 0.4096737504005432


ExecutionResult(
    component_id: Tuner
    execution_id: 15
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={})
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}))

## 11. Trainer: Model Training


In [15]:
%%writefile {TRAINER_MODULE_FILE}
import os
import tensorflow as tf
import tensorflow_transform as tft
from tensorflow.keras import layers
from tfx.components.trainer.fn_args_utils import FnArgs

NUMERICAL_FEATURES = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]
LABEL_KEY = "Class"

def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"

def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs=None,
             batch_size=128) -> tf.data.Dataset:
    """Get post-transform features & create batches of data"""
    
    # Get post-transform feature spec
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    # create batches of data
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    
    return dataset

# Model builder using a dictionary of hyperparameters
def model_builder(hp):
    """Build machine learning model"""
    # Use hyperparameter values
    num_layers = hp.get('num_layers', 2)
    dense_units = hp.get('dense_units', 64)
    dropout_rate = hp.get('dropout_rate', 0.2)
    learning_rate = hp.get('learning_rate', 1e-3)

    # Inputs for all transformed numerical features
    inputs = {}
    for key in NUMERICAL_FEATURES:
        inputs[transformed_name(key)] = tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.float32)

    # Concatenate features
    x = tf.keras.layers.concatenate(list(inputs.values()))

    # Build dense layers
    for _ in range(num_layers):
        x = layers.Dense(dense_units, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout_rate)(x)

    # Output layer (sigmoid activation for binary classification)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model

def _get_serve_tf_examples_fn(model, tf_transform_output):
    """Returns a function that parses raw TF.Examples and runs model inference."""
    model.tft_layer = tf_transform_output.transform_features_layer()
    
    @tf.function
    def serve_tf_examples_fn(serialized_tf_examples):
        feature_spec = tf_transform_output.raw_feature_spec()
        feature_spec.pop(LABEL_KEY)
        
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        transformed_features = model.tft_layer(parsed_features)
        
        return model(transformed_features)
        
    return serve_tf_examples_fn

# Trainer function
def run_fn(fn_args: FnArgs) -> None:
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)

    # Determine steps
    train_steps = fn_args.train_steps if fn_args.train_steps and fn_args.train_steps > 0 else None
    eval_steps = fn_args.eval_steps if fn_args.eval_steps and fn_args.eval_steps > 0 else None

    # Load datasets (repeat if steps are specified, otherwise load once per epoch)
    train_set = input_fn(
        fn_args.train_files,
        tf_transform_output,
        num_epochs=None if train_steps else 1,
        batch_size=128
    )
    val_set = input_fn(
        fn_args.eval_files,
        tf_transform_output,
        num_epochs=None if eval_steps else 1,
        batch_size=128
    )

    # Extract tuner hyperparameters if available
    if fn_args.hyperparameters and 'values' in fn_args.hyperparameters:
        hp = fn_args.hyperparameters['values']
    else: 
        hp = {
            'num_layers': 2,
            'dense_units': 64,
            'dropout_rate': 0.2,
            'learning_rate': 1e-3
        }

    model = model_builder(hp)
    model.summary()

    log_dir = os.path.join(os.path.dirname(fn_args.serving_model_dir), 'logs')
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, update_freq='batch')
    
    es = tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', verbose=1, patience=5)
    
    # Save checkpoint weights to a temp directory
    checkpoint_dir = os.path.join(fn_args.serving_model_dir, 'checkpoint')
    mc = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(checkpoint_dir, 'best_weights'),
        monitor='val_auc',
        mode='max',
        verbose=1,
        save_best_only=True,
        save_weights_only=True
    )

    # Train model
    model.fit(
        x=train_set,
        validation_data=val_set,
        epochs=10,
        steps_per_epoch=train_steps,
        validation_steps=eval_steps,
        callbacks=[tensorboard_callback, es, mc]
    )

    # Load best weights before exporting
    try:
        model.load_weights(os.path.join(checkpoint_dir, 'best_weights'))
        print("Successfully loaded best weights from checkpoint.")
    except Exception as e:
        print(f"Could not load best weights from checkpoint: {e}. Saving final epoch model.")

    # Save final model with serving signature
    signatures = {
        'serving_default': _get_serve_tf_examples_fn(model, tf_transform_output).get_concrete_function(
            tf.TensorSpec(shape=[None], dtype=tf.string, name='examples')
        )
    }

    model.save(fn_args.serving_model_dir, save_format='tf', signatures=signatures)

Overwriting modules/cc_fraud_trainer.py


In [16]:
trainer = Trainer(
    module_file=os.path.abspath(TRAINER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=500),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=100)
)
interactive_context.run(trainer)

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 V1_xf (InputLayer)             [(None, 1)]          0           []                               
                                                                                                  
 V2_xf (InputLayer)             [(None, 1)]          0           []                               
                                                                                                  
 V3_xf (InputLayer)             [(None, 1)]          0           []                               
                                                                                                  
 V4_xf (InputLayer)             [(None, 1)]          0           []                               
                                                                                            

INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: cc_fraud_pipeline_artifacts\cc-fraud-pipeline\Trainer\model\16\Format-Serving\assets


INFO:tensorflow:Assets written to: cc_fraud_pipeline_artifacts\cc-fraud-pipeline\Trainer\model\16\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 16
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={})
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}))

## 12. Model Analysis and Validation
### 12.1 Resolver: Find Best Historical Model


In [17]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('Latest_blessed_model_resolver')

interactive_context.run(model_resolver)

ExecutionResult(
    component_id: Latest_blessed_model_resolver
    execution_id: 17
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={})
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}))

### 12.2 Evaluator: Evaluate Model with TFMA


In [18]:
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='Class')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name='ExampleCount'),
            tfma.MetricConfig(class_name='AUC'),
            tfma.MetricConfig(class_name='FalsePositives'),
            tfma.MetricConfig(class_name='TruePositives'),
            tfma.MetricConfig(class_name='FalseNegatives'),
            tfma.MetricConfig(class_name='TrueNegatives'),
            tfma.MetricConfig(class_name='BinaryAccuracy',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={'value': 0.5}),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={'value': 0.0001})
                )
            )
        ])
    ]
)

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)

interactive_context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 18
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={})
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}))

### 12.3 Visualize Evaluation Results

In [19]:
eval_result = evaluator.outputs['evaluation'].get()[0].uri
tfma_result = tfma.load_eval_result(eval_result)
tfma.view.render_slicing_metrics(tfma_result)
print(tfma_result)

EvalResult(slicing_metrics=[((), {'': {'': {'accuracy': {'doubleValue': 0.999282956123352}, 'auc': {'doubleValue': 0.9771442401220711}, 'precision': {'doubleValue': 0.8292682766914368}, 'recall': {'doubleValue': 0.7157894968986511}, 'loss': {'doubleValue': 0.003881686832755804}, 'example_count': {'doubleValue': 57180.0}, 'false_positives': {'doubleValue': 14.0}, 'true_positives': {'doubleValue': 68.0}, 'false_negatives': {'doubleValue': 27.0}, 'true_negatives': {'doubleValue': 57071.0}, 'binary_accuracy': {'doubleValue': 0.9992829660720531}}}})], plots=[((), None)], attributions=[((), None)], config=model_specs {
  label_key: "Class"
}
slicing_specs {
}
metrics_specs {
  metrics {
    class_name: "ExampleCount"
  }
  metrics {
    class_name: "AUC"
  }
  metrics {
    class_name: "FalsePositives"
  }
  metrics {
    class_name: "TruePositives"
  }
  metrics {
    class_name: "FalseNegatives"
  }
  metrics {
    class_name: "TrueNegatives"
  }
  metrics {
    class_name: "BinaryAccuracy

## 13. Pusher: Export Model for Serving


In [20]:
import os

SERVING_MODEL_DIR = os.path.abspath(os.path.join('serving_model_dir', MODEL_NAME))

pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)

interactive_context.run(pusher)


ExecutionResult(
    component_id: Pusher
    execution_id: 19
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}))

## 14. Verify Exported Serving Model


In [21]:
print("Exported serving model paths:")
if os.path.exists(SERVING_MODEL_DIR):
    for root, dirs, files in os.walk(SERVING_MODEL_DIR):
        print(f"{root} - {dirs} - {files}")
else:
    print("Model serving directory does not exist or Pusher was not triggered (model was not blessed).")

Exported serving model paths:
c:\Users\mfath\Downloads\pipeline\serving_model_dir\cc-fraud-model - ['1787422092'] - []
c:\Users\mfath\Downloads\pipeline\serving_model_dir\cc-fraud-model\1787422092 - ['assets', 'checkpoint', 'variables'] - ['keras_metadata.pb', 'saved_model.pb']
c:\Users\mfath\Downloads\pipeline\serving_model_dir\cc-fraud-model\1787422092\assets - [] - []
c:\Users\mfath\Downloads\pipeline\serving_model_dir\cc-fraud-model\1787422092\checkpoint - [] - ['best_weights.data-00000-of-00001', 'best_weights.index', 'checkpoint']
c:\Users\mfath\Downloads\pipeline\serving_model_dir\cc-fraud-model\1787422092\variables - [] - ['variables.data-00000-of-00001', 'variables.index']
